# 02. Lag effect 검정
가설3 검정


## Yield source note

기본 권장 yield는 `data/processed/eth_yield_panel.csv`의 Lido wstETH protocol exchange-rate 기반 `stake_yield`입니다.
이는 validator gross yield가 아니라 리테일 투자자가 접근 가능한 Lido LST yield proxy입니다.
stETH/ETH 또는 wstETH/ETH 시장가격 디스카운트/프리미엄은 yield가 아니라 별도 basis/depeg risk 변수로 다뤄야 합니다.


In [ ]:
import sys
from pathlib import Path


def resolve_project_root(start: Path) -> Path:
    """Find project root containing `common` and `data` directories."""
    for candidate in [start, *start.parents]:
        if (candidate / "common").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError("Could not resolve project root from current working directory")


PROJECT_ROOT = resolve_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from common.stats import ols_hac
from common.transforms import build_spread, funding_to_daily_annualized, log_return
from common.visualization import plot_timeseries


In [ ]:
RAW = PROJECT_ROOT / 'data' / 'raw'
PROCESSED = PROJECT_ROOT / 'data' / 'processed'

# funding 파일: 신규(perp panel) 포맷 우선, 구 포맷 fallback
eth_f_path = next((p for p in [
    RAW / 'binance_funding_ETHUSDT.csv',
    RAW / 'bybit_funding_ETHUSDT.csv',
    RAW / 'binance_ethusdt_funding.csv',
    RAW / 'bybit_ethusdt_funding.csv',
] if p.exists()), None)
btc_f_path = next((p for p in [
    RAW / 'binance_funding_BTCUSDT.csv',
    RAW / 'bybit_funding_BTCUSDT.csv',
    RAW / 'binance_btcusdt_funding.csv',
    RAW / 'bybit_btcusdt_funding.csv',
] if p.exists()), None)

assert eth_f_path and btc_f_path, 'funding 파일이 없습니다. scripts/collect_perp_funding_panel.py 먼저 실행하세요.'

eth_f = pd.read_csv(eth_f_path)
btc_f = pd.read_csv(btc_f_path)

# 가격 파일: 신규(perp panel) 포맷 우선, 구 포맷 fallback
eth_p_path = next((p for p in [RAW / 'binance_klines_ETHUSDT_1d.csv', RAW / 'binance_ethusdt_1d.csv'] if p.exists()), None)
btc_p_path = next((p for p in [RAW / 'binance_klines_BTCUSDT_1d.csv', RAW / 'binance_btcusdt_1d.csv'] if p.exists()), None)
assert eth_p_path and btc_p_path, '1d 가격 파일이 없습니다. scripts/collect_perp_funding_panel.py 먼저 실행하세요.'
eth_p = pd.read_csv(eth_p_path)
btc_p = pd.read_csv(btc_p_path)

# staking yield 패널(권장): build_eth_yield_panel 출력 우선
st_path = next((p for p in [
    PROCESSED / 'eth_yield_panel.csv',
    PROCESSED / 'lido_wsteth_share_rate.csv',
    RAW / 'eth_staking_yield_daily.csv',
] if p.exists()), None)
assert st_path is not None, 'eth_yield_panel.csv 또는 lido_wsteth_share_rate.csv 또는 eth_staking_yield_daily.csv 가 필요합니다.'
st = pd.read_csv(st_path)


In [ ]:
# 1) funding -> daily annualized
eth_fd = funding_to_daily_annualized(eth_f, time_col='fundingTime', rate_col='fundingRate')
btc_fd = funding_to_daily_annualized(btc_f, time_col='fundingTime', rate_col='fundingRate')

# 2) spread 생성
spread = build_spread(eth_fd, btc_fd, value_col='funding_ann', out_col='spread')


def normalize_daily_date(series: pd.Series) -> pd.Series:
    """Normalize mixed date/timestamp inputs to naive daily datetime."""
    return pd.to_datetime(series, utc=True, errors='coerce').dt.tz_convert(None).dt.normalize()


# 3) 가격 수익률 생성
for px in (eth_p, btc_p):
    px['date'] = normalize_daily_date(px['date'])
eth_p['ret_eth'] = log_return(eth_p['close'])
btc_p['ret_btc'] = log_return(btc_p['close'])
ret = eth_p[['date','ret_eth']].merge(btc_p[['date','ret_btc']], on='date', how='inner')
ret['ret_eth_btc'] = ret['ret_eth'] - ret['ret_btc']

# 단순 RV proxy (절대수익률)
ret['rv_eth_btc'] = (ret['ret_eth'].abs() - ret['ret_btc'].abs())

# staking
st['date'] = normalize_daily_date(st['date'])
st['stake_yield'] = pd.to_numeric(st['stake_yield'], errors='coerce')
# stake_yield가 % 값이면 100으로 나누기 (예: 3.5 -> 0.035)
if st['stake_yield'].dropna().median() > 1:
    st['stake_yield'] = st['stake_yield'] / 100.0

spread['date'] = normalize_daily_date(spread['date'])
df = spread[['date','spread']].merge(ret[['date','ret_eth_btc','rv_eth_btc']], on='date', how='inner').merge(st[['date','stake_yield']], on='date', how='inner').dropna().sort_values('date').reset_index(drop=True)

df.head(), df.describe().T[['mean','std','min','max']]


In [ ]:
# lag / lead 생성
lag_df = df.copy()
lag_df['stake_yield_l1'] = lag_df['stake_yield'].shift(1)
lag_df['ret_eth_btc_l1'] = lag_df['ret_eth_btc'].shift(1)
lag_df['rv_eth_btc_l1'] = lag_df['rv_eth_btc'].shift(1)
lag_df['spread_f1'] = lag_df['spread'].shift(-1)
lag_df = lag_df.dropna().reset_index(drop=True)
lag_df[['date','spread','spread_f1','stake_yield','stake_yield_l1']].head()


In [ ]:
# 모델 A: spread_t ~ stake_yield_{t-1} + controls_{t-1}
resA = ols_hac(lag_df['spread'], lag_df[['stake_yield_l1','ret_eth_btc_l1','rv_eth_btc_l1']], maxlags=5)
print(resA.summary())


In [ ]:
# 모델 B: spread_{t+1} ~ stake_yield_t + controls_t
resB = ols_hac(lag_df['spread_f1'], lag_df[['stake_yield','ret_eth_btc','rv_eth_btc']], maxlags=5)
print(resB.summary())


In [ ]:
# lag 1/2/3 민감도
rows=[]
for L in [1,2,3]:
    t= df.copy()
    t[f'stake_l{L}']=t['stake_yield'].shift(L)
    t[f'ret_l{L}']=t['ret_eth_btc'].shift(L)
    t[f'rv_l{L}']=t['rv_eth_btc'].shift(L)
    t=t.dropna()
    r=ols_hac(t['spread'], t[[f'stake_l{L}',f'ret_l{L}',f'rv_l{L}']], maxlags=5)
    rows.append({'lag':L, 'coef':r.params[f'stake_l{L}'], 'pvalue':r.pvalues[f'stake_l{L}']})
pd.DataFrame(rows)
